In [1]:
import warnings
warnings.filterwarnings('ignore')

# 고급 검색 기법의 이해 및 활용

<img src="Advanced_Retrieval.png" width="700" align="left" />

# LangSmith

LangSmith는 LangChain 개발 팀에서 제작한 LLM 애플리케이션 전용 디버깅, 관측성(Observability), 평가(Evaluation) 및 모니터링 플랫폼으로 AI 에이전트나 RAG 시스템처럼 복잡한 LLM 워크프로를 시각화하고 최적화하는 데 필수적인 도구이다.

LangSmith 사이트에 접속해서 회원 가입 후 API key를 발급받는다.

<img src="langSmith1.png" width="1200" align="left" />

<img src="langSmith2.png" width="1200" align="left" />

## .env 파일에 LangSmith 설정하기

LANGCHAIN_TRACING_V2=true  
LANGCHAIN_ENDPOINT="https://api.smith.langchain.com"  
LANGCHAIN_API_KEY="API key"  
LANGCHAIN_PROJECT="프로젝트 이름"

## LangSmith 설정 세부 설명

`LANGCHAIN_TRACING_V2`  
&nbsp;&nbsp;&nbsp;▶ LangChain의 2세대 추적 기능의 활성 여부를 지정한다. 민감한 데이터에는 사용하지 않는다.  
&nbsp;&nbsp;&nbsp;▶ 이 값이 true로 설정되어 있어야만 코드 실행 시 발생하는 모든 프롬프트, 응답, 도구 호출 등의 로그가 자동으로 기록된다.  
`LANGCHAIN_ENDPOINT`  
&nbsp;&nbsp;&nbsp;▶ 데이터를 보낼 LangSmith 서버 주소를 입력한다.  
&nbsp;&nbsp;&nbsp;▶ LangChain 라이브러리가 데이터를 어디로 전송할지 결정한다. 특별히 구축한 서버를 쓰지 않는 한 https://api.smith.langchain.com 를 사용한다.  
`LANGCHAIN_API_KEY`  
&nbsp;&nbsp;&nbsp;▶ LangSmith 서비스에 접근하기 위한 고유 인증 키를 입력한다. 유출되지 않도록 주의한다.  
`LANGCHAIN_PROJECT`  
&nbsp;&nbsp;&nbsp;▶ LangSmith 서버로 전송되는 데이터를 분류할 프로젝트 이름을 지정한다.  
&nbsp;&nbsp;&nbsp;▶ LangSmith 대시보드에서 여러 개의 앱을 개발할 때, 로그가 섞이지 않도록 구분하는 역할을 한다. 생략시 default라는 이름을 사용한다.

# 환경 설정

## 기본 라이브러리

In [20]:
import os, json, re
from glob import glob
from pprint import pprint
import numpy as np
import pandas as pd

from langchain_community.document_loaders import TextLoader
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.document_loaders import JSONLoader
from langchain_core.documents import Document
from langchain_text_splitters import CharacterTextSplitter
from langchain_text_splitters import RecursiveCharacterTextSplitter
from transformers import AutoTokenizer
from langchain_openai import OpenAIEmbeddings
from langchain_huggingface.embeddings import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_core.prompts import PromptTemplate
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser

## .env 환경 변수

In [3]:
from dotenv import load_dotenv
load_dotenv()

# LangSmith 추적 여부 확인하기(true: LangSmith 추적 활성화, false: LangSmith 추적 비활성화)
print('LangSmith 추적 여부:', os.getenv('LANGCHAIN_TRACING_V2'))

LangSmith 추적 여부: true


# 쿼리 확장

사용자의 원래 쿼리를 확장하여 더 관련성 있는 결과를 얻는 기술이다.

<img src="query.png" width="1200" align="left" />

# 벡터저장소 로드

이미 생성된 Chroma 벡터저장소를 불러온다.

In [4]:
# 임베딩 모델을 생성한다.
# Chroma 벡터저장소를 저장할 때 썼던 모델과 불러올 때 쓰는 모델이 반드시 같아야 검색이 정확하게 작동된다.
embeddings_model = HuggingFaceEmbeddings(model='BAAI/bge-m3')

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

In [5]:
# Chroma 벡터저장소를 불러온다.
chroma_db = Chroma(
    embedding_function=embeddings_model,
    collection_name='hf_bge_m3',
    persist_directory='./chroma_db'
)

불러온 Chroma 벡터저장소를 벡터검색기로 만든다.

In [6]:
chroma_k_retriever = chroma_db.as_retriever(search_kwargs={'k': 2})

벡터검색기에서 질문을 던져서 검색기를 실행한다. RAG와 관련된 무엇인가를 실행하면 정보가 LangSmith로 전달된다.

In [7]:
query = '테슬라의 성장 동력은 무엇인가요?'
retriever_docs = chroma_k_retriever.invoke(query)

<img src="langSmith3.png" width="1200" align="left" />

<img src="langSmith4.png" width="1200" align="left" />

<img src="langSmith5.png" width="1200" align="left" />

In [14]:
print(f'쿼리: {query}')
print('검색 결과')
for doc in retriever_docs:
    print(doc.page_content)
    print(doc.metadata)
    print('-' * 100)

쿼리: 테슬라의 성장 동력은 무엇인가요?
검색 결과
머스크는 최대 주주이자 회장으로서 회사를 현재의 성공으로 이끌었습니다
회사 이름은 유명한 물리학자이자 전기공학자인 니콜라 테슬라의 이름을 따서 지어졌습니다
테슬라는 2010년 6월 나스닥에 상장되었습니다
2023년 테슬라는 1,808,581대의 차량을 판매하여 2022년에 비해 37
65% 증가했습니다

(참고: 이 문서는 리비안에 대한 정보를 담고 있습니다.)
{'source': './data\\테슬라_KR.txt', 'doc_id': 4}
----------------------------------------------------------------------------------------------------
테슬라(Tesla, Inc
)는 텍사스주 오스틴에 본사를 둔 미국의 대표적인 전기차 제조업체입니다
2003년 마틴 에버하드(CEO)와 마크 타페닝(CFO)에 의해 설립된 테슬라는 2004년 페이팔과 Zip2의 공동 창업자인 일론 머스크의 참여로 큰 전환점을 맞았습니다

(참고: 이 문서는 리비안에 대한 정보를 담고 있습니다.)
{'doc_id': 3, 'source': './data\\테슬라_KR.txt'}
----------------------------------------------------------------------------------------------------


# 쿼리 확장 - 멀티 쿼리(Multi Query) 기법

멀티 쿼리 기법은 사용자의 모호한 질문을 LLM이 다양한 관점의 여러 질문으로 재구성하여 검색의 범위를 넓히는(정확도를 높이는) 전략이다. `내가 개떡같이 말해도 찰떡같이 알아듣게` 만들기 위해 하나의 질문을 여러 개로 만드는 것이다.

작동 순서  
① `질문 변형`: 사용자가 질문을 던지면, 내부의 LLM이 이 질문과 의미적으로 유사하거나 보안된 질문들을 새로 생성한다.  
② `개별 벡터 검색`: 생성된 모든 질문(원본 + 변형 질문들)을 각각 벡터저장소에서 조회하여 관련 문서들을 가져온다.  
③ `결과 통합 및 중복 제거`: 각 질문이 찾아온 문서들을 하나로 합친 뒤, 중복되는 문서들은 제거하여 고유한 문서 집합을 만든다.  
④ `최종 답변 생성`: 취합된 풍부한 문서 데이터를 바탕으로 LLM이 최종 답변을 작성한다.

장점  
`검색 누락 방지`: 사용자가 키워드를 정확히 입력하지 않더라도, LLM이 유의어나 관련 전문 용어로 질문을 바꿔주기 때문에 관련 문서를 찾을 확률이 높아진다.  
`언어적 차이 극복`: 구어체 질문을 문서 데이터에 적합한 문어체나 기술적인 질문으로 변환하여 검색 효율을 극대화한다.  
`입체적 정보 수집`: 질문을 다각도로 쪼개어 검색하므로, 한 방향의 검색으로는 얻기 힘든 포괄적인 배경 지식을 수집할 수 있다.

단점  
`비용 및 속도`: 검색 전에 LLM을 한 번 거쳐야 하므로 API 호출 비용이 발생하고, 답변이 나오기 까지 대기 시간이 길어진다.  
`노이즈 유입`: 질문을 너무 넓게 확장하면 원래 의도와 관련 없는 '불필요한 정보'가 섞여 들어와 답변의 초점을 흐릴 수 있다.  
`컨텍스트 과부하`: 너무 많은 문서가 검색될 경우, LLM이 한 번에 처리해야 할 텍스트 양이 많아져 핵심 내용을 놓치거나 연산량이 늘어난다.

## MultiQueryRetriever

사용자의 질문을 다양한 관점으로 재해석(멀티 쿼리)하고, 이를 통해 더 풍부한 검색 결과를 얻기위해 MultiQueryRetriever를 import 한다.

In [15]:
from langchain.retrievers.multi_query import MultiQueryRetriever

In [16]:
# 멀티 쿼리 리트리버에 사용할 LLM 모델을 생성한다.
llm = ChatOpenAI(model='gpt-4o-mini', temperature=0.7, max_completion_tokens=100)

멀티 쿼리 리트리버를 생성한다.

In [17]:
# from_llm() 메소드로 리트리버와 LLM 모델을 넘겨서 멀티 쿼리 리트리버를 생성한다.
multi_query_retriever = MultiQueryRetriever.from_llm(
    # 이미 생성되어 있는 벡터검색기를 넣어준다.
    retriever=chroma_k_retriever,
    # 질문을 여러 개로 변형할 때 사용할 LLM 언어 모델을 넣어준다.
    llm=llm
)

멀티 쿼리 리트리버에 질문을 던져서 멀티 쿼리 리트리버를 실행한다.

In [18]:
query = '테슬라의 성장 동력은 무엇인가요?'
retriever_docs = multi_query_retriever.invoke(query)

<img src="langSmith6.png" width="1200" align="left" />

<img src="langSmith7.png" width="1200" align="left" />

In [19]:
print(f'쿼리: {query}')
print('검색 결과')
for doc in retriever_docs:
    print(doc.page_content)
    print(doc.metadata)
    print('-' * 100)

쿼리: 테슬라의 성장 동력은 무엇인가요?
검색 결과
머스크는 최대 주주이자 회장으로서 회사를 현재의 성공으로 이끌었습니다
회사 이름은 유명한 물리학자이자 전기공학자인 니콜라 테슬라의 이름을 따서 지어졌습니다
테슬라는 2010년 6월 나스닥에 상장되었습니다
2023년 테슬라는 1,808,581대의 차량을 판매하여 2022년에 비해 37
65% 증가했습니다

(참고: 이 문서는 리비안에 대한 정보를 담고 있습니다.)
{'source': './data\\테슬라_KR.txt', 'doc_id': 4}
----------------------------------------------------------------------------------------------------
테슬라(Tesla, Inc
)는 텍사스주 오스틴에 본사를 둔 미국의 대표적인 전기차 제조업체입니다
2003년 마틴 에버하드(CEO)와 마크 타페닝(CFO)에 의해 설립된 테슬라는 2004년 페이팔과 Zip2의 공동 창업자인 일론 머스크의 참여로 큰 전환점을 맞았습니다

(참고: 이 문서는 리비안에 대한 정보를 담고 있습니다.)
{'doc_id': 3, 'source': './data\\테슬라_KR.txt'}
----------------------------------------------------------------------------------------------------


## Custom Prompt

LLM이 자동으로 만들어주는 프롬프트를 사용하지 않고 질문의 개수를 바꾸거나 추가적인 요청 사항이 있을 경우 사용자가 직접 프롬프트를 만들어 사용한다.

파이썬의 타입 힌트 기능으로 함수의 리턴 타입이 리스트 형태라는 것을 명시적으로 알려주기 위해서 List를 import 한다.

In [54]:
from typing import List

AI가 답변한 내용(문자열)을 우리가 원하는 형태(JSON, Dict, List 등)로 가공하기 위한 출력 파서의 기본 틀을 사용하기 위해 BaseOutputParser를 import 한다.

In [55]:
from langchain_core.output_parsers import BaseOutputParser

멀티 쿼리를 만드는 커스텀 프롬프트를 만든다.

In [56]:
template = '''
당신은 AI 언어 모델 보조자입니다.
벡터저장소에서 관련 문서를 검색하기 위해, 주어진 사용자 질문을 네 가지 다른 버전으로 생성하세요.
목표는 거리 기반 유사도 검색의 한계를 극복하기 위해 질문을 다양한 관점에서 재구성하는 것입니다.

생성된 질문들은 다음과 같은 특징을 갖추어야 합니다.
1. 원문 질문의 핵심 의도는 유지하되, 다른 표현이나 관점을 사용하세요.
2. 가능한 경우 유의어나 관련 개념을 포함하세요.
3. 다양하고 관련 있는 정보가 포함될 수 있도록 질문의 범위를 약간 넓히거나 좁히세요.

각 질문을 새 줄에 작성하고, 질문 내용만 포함하세요.

[원래 질문]
{question}

[대체 질문]
'''

custom_prompt = PromptTemplate(
    input_variables=['question'],
    template=template
)

출력 파서의 기본 틀(BaseOutputParser)

LLM이 출력한 줄글 형태의 답변을 파이썬 리스트로 변형하는 출력 파서 객체를 LangChain에서 제공하는 기본 파서 틀을 상속받고 결과값으로 문자열들이 저장된 리스트를 내보내겠다고 만든다.

In [64]:
class LineListOutputParser(BaseOutputParser):
    # LLM이 출력한 텍스트를 받아서 문자열이 저장된 리스트로 가공하는 메소드를 선언한다.
    # 클래스에서 선언하는 모든 메소드의 첫 번째 인수는 무조건 'self'가 나와야 한다.
    # LLM의 답변을 text 변수에 받아서 답변을 가공해서 문자열이 저장된 리스트 타입으로 리턴하는 메소드로 이름은 무조건 'parse'를 사용해야 한다.
    def parse(self, text: str) -> List[str]:
        # text.strip().split('\n'): LLM 답변의 앞뒤에 붙은 불필요한 공백을 제거하고 '\n' 기준으로 나눠서 리스트로 만든다.
        # for line in ...: text.strip().split('\n')를 실행해서 만들어진 리스트의 요소를 line에 저장하며 반복한다.
        # line.strip(): text.strip().split('\n')를 실행해서 만들어진 리스트의 각 요소들에서 앞뒤의 불필요한 빈 칸을 제거한다.
        # if line.strip(): line.strip()를 실행해서 앞뒤의 공백을 제거했는데 내용이 없는 빈 줄이라면 리스트에 넣지 않는다. 필터링
        return [line.strip() for line in text.strip().split('\n') if line.strip()]

RAG 체인을 구성한다.

In [65]:
rag_chain = custom_prompt | llm | LineListOutputParser()

RAG 체인을 실행한다.

In [66]:
query = '테슬라의 성장 동력은 무엇인가요?'
result = rag_chain.invoke({'question': query})

<img src="langSmith8.png" width="1200" align="left" />

In [67]:
print('생성된 대안 질문들')
for index, item in enumerate(result, start=1):
    print(f'{index}. {item}')

생성된 대안 질문들
1. 테슬라의 발전을 이끄는 주요 요인은 무엇인가요?
2. 테슬라의 성공을 가능하게 하는 핵심 요소는 무엇입니까?
3. 테슬라의 비즈니스 확장에서 중요한 역할을 하는 요인은 어떤 것들이 있을까요?
4. 테슬사가 지속적으로 성장할 수 있는 이유는 무엇인지 알고 싶습니다.
